In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "457_build_gold_vw_fact_project_bid.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.fact_bid_project"
TARGET_VIEW = f"{catalog}.gold.vw_fact_project_bid"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_fact_project_bid
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    SELECT
          bid_project_fact_key                        AS `Bid Project Fact Key`
        , project_key                                 AS `Project Key`
        , vendor_key                                  AS `Vendor Key`

        , project_classification_key                  AS `Project Classification Key`
        , county_key                                  AS `County Key`

        , project_id                                  AS `Project ID`
        , vendor_name                                 AS `Vendor Name`
        , vendor_name_standardized                    AS `Vendor Name Standardized`
        , project_name                                AS `Project Name`
        , project_number                              AS `Project Number`
        , state_project_number                        AS `State Project Number`
        , control_section_job_csj                     AS `Control Section Job CSJ`
        , controlling_project_id_ccsj                 AS `Controlling Project ID CCSJ`
        , project_type                                AS `Project Type`
        , project_classification                      AS `Project Classification`
        , CAST(project_actual_let_date AS DATE)       AS `Project Actual Let Date`
        , project_estimated_let_date                  AS `Project Estimated Let Date`
        , project_limits_from                         AS `Project Limits From`
        , project_limits_to                           AS `Project Limits To`
        , county                                      AS `County`
        , location                                    AS `Location`
        , district_division                           AS `District Division`
        , highway                                     AS `Highway`
        , short_description                           AS `Short Description`
        , federal_project_number                      AS `Federal Project Number`

        , bid_item_count                              AS `Bid Item Count`
        , distinct_item_specification_count           AS `Distinct Item Specification Count`
        , distinct_bid_item_sequence_count            AS `Distinct Bid Item Sequence Count`
        , distinct_submit_sequence_count              AS `Distinct Submit Sequence Count`
        , total_bid_item_quantity                     AS `Total Bid Item Quantity`
        , bid_item_extended_total                     AS `Bid Item Extended Total`
        , min_bid_total_amount                        AS `Min Bid Total Amount`
        , max_bid_total_amount                        AS `Max Bid Total Amount`
        , has_conflicting_bid_total_amounts           AS `Has Conflicting Bid Total Amounts`
        , bid_total_vs_item_rollup_abs_diff           AS `Bid Total Vs Item Rollup Abs Diff`
        , min_bid_rank_sequence_number                AS `Min Bid Rank Sequence Number`
        , max_bid_rank_sequence_number                AS `Max Bid Rank Sequence Number`
        , has_any_low_bidder_items                    AS `Has Any Low Bidder Items`
        , is_low_bidder                               AS `Is Low Bidder`
        , payment_adjustment_item_count               AS `Payment Adjustment Item Count`
        , special_deduction_item_count                AS `Special Deduction Item Count`
        , nonstandard_adjustment_item_count           AS `Nonstandard Adjustment Item Count`
        , vendor_project_has_resubmission_history     AS `Vendor Project Has Resubmission History`
        , any_item_has_resubmission_history           AS `Any Item Has Resubmission History`
        , any_item_changed_across_submit_sequences    AS `Any Item Changed Across Submit Sequences`
        , any_item_missing_from_latest_project_submit AS `Any Item Missing From Latest Project Submit`
        , any_item_changed_value_across_submits       AS `Any Item Changed Value Across Submits`
        , project_bid_rank                            AS `Project Bid Rank`
        , bidder_count_in_project                     AS `Bidder Count In Project`
        , lowest_bid_amount_in_project                AS `Lowest Bid Amount In Project`
        , highest_bid_amount_in_project               AS `Highest Bid Amount In Project`
        , bid_spread_amount_in_project                AS `Bid Spread Amount In Project`
        , bid_vs_low_bid_ratio                        AS `Bid Vs Low Bid Ratio`
        , latest_source_updated_at                    AS `Latest Source Updated At`
        , latest_ingested_at_utc                      AS `Latest Load Timestamp`
    FROM {SOURCE_TABLE}
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_VIEW}").collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_fact_project_bid successfully.")

    print("=" * 70)
    print("Built gold.vw_fact_project_bid")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise